In [0]:
%run ../includes/storage_config

In [0]:
from utils.common_functions import merge_data

Select fields:
1. date
2. morning peak / off peak / evening peak
3. avg_arrival_delay 
4. avg_departure_delay 
5. total arrivals
6. arrival delay rate (with 5-minute error margin)
7. total departures
8. departure delay rate (with 5-minute error margin)



In [0]:
delay_pattern_by_data_sql = f'''
  SELECT 
    service_date AS date,
    CASE 
            WHEN weekday(service_date) >= 5 THEN "Off Peak"
            WHEN (hour(COALESCE(stop_departure_time, stop_arrival_time)) >= 6 AND minute(COALESCE(stop_departure_time, stop_arrival_time)) >= 30)
                  OR hour(COALESCE(stop_departure_time, stop_arrival_time)) BETWEEN 7 and 8
                  OR (hour(COALESCE(stop_departure_time, stop_arrival_time)) = 9 AND minute(COALESCE(stop_departure_time, stop_arrival_time)) = 0)
            THEN "Morning Peak"
            WHEN hour(COALESCE(stop_departure_time, stop_arrival_time)) BETWEEN 16 AND 17
                  OR (hour(COALESCE(stop_departure_time, stop_arrival_time)) = 19 AND minute(COALESCE(stop_departure_time, stop_arrival_time)) <= 30)
            THEN "Evening Peak"
            ELSE "Off Peak"
        END AS time_period,
    ROUND(AVG(stop_arrival_delay),2) as avg_arrival_delay,
    ROUND(AVG(stop_departure_delay),2) as avg_depature_delay,
    COUNT(stop_arrival_delay) AS total_arrivals,
    COUNT(stop_departure_delay) AS total_departures,
    ROUND(COUNT(CASE WHEN stop_arrival_delay > 5 THEN 1 END)/ NULLIF(COUNT(stop_arrival_delay),0),2) AS arrival_delay_rate,
    ROUND(COUNT(CASE WHEN stop_departure_delay > 5 THEN 1 END)/ NULLIF(COUNT(stop_departure_delay),0),2) AS departure_delay_rate
  FROM {processed_table_name_stops}
  WHERE stop_arrival_delay IS NOT NULL OR stop_departure_delay IS NOT NULL
  GROUP BY 1 , 2
  ORDER BY date, 
        CASE time_period
              WHEN 'Morning Peak' THEN 1
              WHEN 'Evening Peak' THEN 2
              WHEN 'Off Peak'     THEN 3
              ELSE 4
          END;
'''

resutls_df = spark.sql(delay_pattern_by_data_sql)
display(resutls_df)

In [0]:
merge_condition = 'target.date = source.date AND target.time_period = source.time_period'
merge_data(
    spark = spark,
    input_df = resutls_df, 
    schema_name= schema,
    table_name= daily_delay_table,
    merge_condition = merge_condition)